# Cuaderno 2: Simulación del tsunami de 1979

Tsunami del 12 de diciembre de 1979 (Mw 8.2, Tumaco) — solver NSWE 2D en **NumPy puro**.

**Runtime:** CPU (no requiere TPU). Tiempo estimado: **< 5 min** en Google Colab.

`Entorno de ejecución` → `Ejecutar todo`

In [ ]:
# Celda 1/8 — Instalar dependencias
%pip install -q rasterio pyproj scipy
print('✓ Dependencias listas')

In [ ]:
# Celda 2/8 — Imports y rutas
import sys, os, time
import numpy as np
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling

IN_COLAB   = 'google.colab' in sys.modules
WORK_DIR   = Path('/content') if IN_COLAB else Path('.')
OUTPUT_DIR = WORK_DIR / 'output_1979'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEM_PATH   = WORK_DIR / 'tumaco_dem_utm_sim.tif'
print(f'✓ Paths configurados | Colab: {IN_COLAB}')

In [ ]:
# Celda 3/8 — Preparar DEM (auto-genera si no existe)
if not DEM_PATH.exists():
    print('DEM no encontrado — generando...')
    LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 0.5, 3.5, -79.5, -77.0
    CRS_SRC  = CRS.from_epsg(4326)
    CRS_DEST = CRS.from_epsg(32618)
    geo_path = WORK_DIR / 'tumaco_bathy_geo.tif'
    bathy_ok = False

    # Intento 1: GEBCO 2023
    try:
        url = (f'https://download.gebco.net/a/gebco_2023'
               f'?lat1={LAT_MIN}&lat2={LAT_MAX}&lon1={LON_MIN}&lon2={LON_MAX}&format=netcdf4')
        r = requests.get(url, timeout=120)
        if r.status_code == 200 and len(r.content) > 5000:
            import xarray as xr, io
            ds      = xr.open_dataset(io.BytesIO(r.content))
            varname = [v for v in ds.data_vars if 'elev' in v.lower() or 'z' in v.lower()][0]
            da      = ds[varname]
            lat_dim = [d for d in da.dims if 'lat' in d.lower()][0]
            lon_dim = [d for d in da.dims if 'lon' in d.lower()][0]
            if da[lat_dim].values[0] < da[lat_dim].values[-1]:
                da = da.isel({lat_dim: slice(None, None, -1)})
            arr  = da.values.astype('float32')
            lats = da[lat_dim].values
            lons = da[lon_dim].values
            tf_g = from_bounds(lons.min(), lats.min(), lons.max(), lats.max(),
                               arr.shape[1], arr.shape[0])
            with rasterio.open(geo_path, 'w', driver='GTiff',
                               height=arr.shape[0], width=arr.shape[1],
                               count=1, dtype='float32', crs=CRS_SRC, transform=tf_g) as dst:
                dst.write(arr, 1)
            bathy_ok = True
            print('  ✓ GEBCO 2023')
    except Exception as e:
        print(f'  ✗ GEBCO: {e}')

    # Intento 2: ETOPO 2022 (NOAA)
    if not bathy_ok:
        try:
            nc = int((LON_MAX - LON_MIN) / 0.004167)
            nr = int((LAT_MAX - LAT_MIN) / 0.004167)
            url2 = ('https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/'
                    'DEM_global_mosaic/ImageServer/exportImage'
                    f'?bbox={LON_MIN},{LAT_MIN},{LON_MAX},{LAT_MAX}'
                    f'&bboxSR=4326&size={nc},{nr}&imageSR=4326&format=tiff&pixelType=F32&f=image')
            r2 = requests.get(url2, timeout=120)
            r2.raise_for_status()
            with open(geo_path, 'wb') as f:
                f.write(r2.content)
            bathy_ok = True
            print('  ✓ ETOPO 2022')
        except Exception as e:
            print(f'  ✗ ETOPO: {e}')

    # Intento 3: DEM sintético
    if not bathy_ok:
        print('  ⚠ DEM sintético (solo pruebas)')
        res    = 0.004167
        lons_s = np.arange(LON_MIN, LON_MAX, res)
        lats_s = np.arange(LAT_MIN, LAT_MAX, res)
        nr, nc = len(lats_s), len(lons_s)
        xr2 = (lons_s - LON_MIN) / (LON_MAX - LON_MIN)
        z   = np.where(xr2 < 0.70, -4000 + 3800*(xr2/0.70),
              np.where(xr2 < 0.88, -200 + 200*((xr2-0.70)/0.18), 10.0))
        dem_s  = np.tile(z, (nr, 1)).astype('float32')
        dem_s += gaussian_filter(np.random.randn(nr, nc)*50, sigma=5)
        tf_s   = from_bounds(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, nc, nr)
        with rasterio.open(geo_path, 'w', driver='GTiff', height=nr, width=nc,
                           count=1, dtype='float32', crs=CRS_SRC, transform=tf_s) as dst:
            dst.write(dem_s, 1)

    # Reproyectar a UTM 18N
    utm_path = WORK_DIR / 'tumaco_dem_utm.tif'
    with rasterio.open(geo_path) as src:
        tf_utm, w_utm, h_utm = calculate_default_transform(
            src.crs, CRS_DEST, src.width, src.height, *src.bounds)
        meta = src.meta.copy()
        meta.update({'crs': CRS_DEST, 'transform': tf_utm,
                     'width': w_utm, 'height': h_utm, 'dtype': 'float32'})
        with rasterio.open(utm_path, 'w', **meta) as dst:
            reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=tf_utm, dst_crs=CRS_DEST,
                      resampling=Resampling.bilinear)

    # Invertir eje X para compatibilidad con el formato de salida
    with rasterio.open(utm_path) as src:
        arr_u  = src.read(1)
        old_tf = src.transform
        meta2  = src.meta.copy()
    arr_flip = arr_u[:, ::-1]
    meta2['transform'] = rasterio.transform.from_origin(
        west=old_tf.c + old_tf.a*arr_u.shape[1], north=old_tf.f,
        xsize=-old_tf.a, ysize=-old_tf.e)
    with rasterio.open(DEM_PATH, 'w', **meta2) as dst:
        dst.write(arr_flip, 1)
    print(f'  ✓ DEM guardado: {DEM_PATH}')

with rasterio.open(str(DEM_PATH)) as src:
    _sh  = (src.height, src.width)
    _res = abs(src.transform.a)
print(f'✓ DEM: {_sh[0]}×{_sh[1]} px | {_res:.0f} m/px')

In [ ]:
# Celda 4/8 — Cargar DEM, grilla de simulación y condiciones iniciales
#
# El DEM almacenado (tumaco_dem_utm_sim.tif) tiene:
#   row 0 = norte, col 0 = este (costa), col N-1 = oeste (océano)
# Para el solver necesitamos x/y crecientes con índice, así que
# hacemos doble flip; al guardar los snapshots hacemos el flip inverso.

with rasterio.open(str(DEM_PATH)) as src:
    dem_stored = src.read(1).astype(np.float64)
    tr         = src.transform
    ny, nx     = dem_stored.shape
    dx         = abs(tr.a)   # m/px en x
    dy         = abs(tr.e)   # m/px en y

# Reorientar: row0=sur, col0=oeste (océano) → x e y crecen con índice
b  = dem_stored[::-1, ::-1]
Lx = nx * dx
Ly = ny * dy
x1d = np.arange(nx) * dx + dx / 2   # [0, Lx]
y1d = np.arange(ny) * dy + dy / 2   # [0, Ly], 0 = sur
XX, _ = np.meshgrid(x1d, y1d)

# Condiciones iniciales: N-wave Carrier (1979, Mw 8.2)
G_GRAV    = 9.81
EPS_DEPTH = 1e-3   # m — profundidad mínima húmeda

ocean_mask = b < -10.0
x_coast    = float(np.max(XX[ocean_mask])) if ocean_mask.any() else 0.28 * Lx
x_epic     = x_coast - 60_000.0   # epicentro ~60 km mar adentro

# Parámetros N-wave (Carrier et al. 2003)
A_NW  = 1.5
LAMB  = 80_000.0
K1    = 28.416 / LAMB**2
K2    = 256.0  / LAMB**2
x1_nw = x_epic + 0.3153 * LAMB   # cresta (elevación)
x2_nw = x_epic                    # depresión frontal

eta0 = 2.0 * (A_NW * np.exp(-K1 * (XX - x1_nw)**2)
              - (A_NW / 3.0) * np.exp(-K2 * (XX - x2_nw)**2))

h0      = np.where(b < 0, -b, EPS_DEPTH)                         # profundidad en reposo
h_init  = np.where(b < 0, np.maximum(h0 + eta0, EPS_DEPTH), EPS_DEPTH)
qx_init = np.zeros((ny, nx))
qy_init = np.zeros((ny, nx))

print(f'DEM: {ny}×{nx} px | dx={dx:.0f}m | {Lx/1e3:.0f}×{Ly/1e3:.0f} km')
print(f'x_costa={x_coast/1e3:.0f} km | x_epicentro={x_epic/1e3:.0f} km')
print(f'η₀_max = {eta0[ocean_mask].max():.2f} m')

In [ ]:
# Celda 5/8 — Solver NSWE 2D (Lax-Friedrichs, NumPy)
def nswe_lf_step(h, qx, qy, b_bed, dx, dy, dt,
                 g=9.81, n_mann=0.025, eps=1e-3):
    """Un paso temporal de las NSWE 2D con esquema Lax-Friedrichs."""
    H = np.maximum(h, eps)

    # Flujos conservativos
    Fx0 = qx;               Fy0 = qy
    Fx1 = qx**2/H + 0.5*g*h**2;  Fy1 = qx*qy/H
    Fx2 = qx*qy/H;          Fy2 = qy**2/H + 0.5*g*h**2

    # Índices de vecinos
    c  = slice(1, -1)
    rp = slice(2, None); rm = slice(0, -2)   # fila ±1
    cp = slice(2, None); cm = slice(0, -2)   # col  ±1

    # Pendiente del fondo (diferencias centradas)
    dbdx = (b_bed[c, cp] - b_bed[c, cm]) / (2 * dx)
    dbdy = (b_bed[rp, c] - b_bed[rm, c]) / (2 * dy)

    # Fricción de Manning
    Hc    = H[c, c]
    speed = np.sqrt(qx[c,c]**2 + qy[c,c]**2)
    fric  = g * n_mann**2 * speed / np.maximum(Hc, eps)**(7/3)
    Sx    = -g * Hc * dbdx - fric * qx[c, c]
    Sy    = -g * Hc * dbdy - fric * qy[c, c]

    # Actualización Lax-Friedrichs
    h_new  = np.copy(h)
    qx_new = np.copy(qx)
    qy_new = np.copy(qy)

    h_new[c,c]  = (0.25*(h[c,cp]+h[c,cm]+h[rp,c]+h[rm,c])
                   - dt/(2*dx)*(Fx0[c,cp]-Fx0[c,cm])
                   - dt/(2*dy)*(Fy0[rp,c]-Fy0[rm,c]))
    qx_new[c,c] = (0.25*(qx[c,cp]+qx[c,cm]+qx[rp,c]+qx[rm,c])
                   - dt/(2*dx)*(Fx1[c,cp]-Fx1[c,cm])
                   - dt/(2*dy)*(Fy1[rp,c]-Fy1[rm,c])
                   + dt * Sx)
    qy_new[c,c] = (0.25*(qy[c,cp]+qy[c,cm]+qy[rp,c]+qy[rm,c])
                   - dt/(2*dx)*(Fx2[c,cp]-Fx2[c,cm])
                   - dt/(2*dy)*(Fy2[rp,c]-Fy2[rm,c])
                   + dt * Sy)

    # Condición de contorno Neumann (gradiente cero en bordes)
    for arr in (h_new, qx_new, qy_new):
        arr[0,:]  = arr[1,:];  arr[-1,:] = arr[-2,:]
        arr[:,0]  = arr[:,1];  arr[:,-1] = arr[:,-2]

    # Celda húmeda/seca
    h_new = np.maximum(h_new, eps)
    dry   = h_new <= eps
    qx_new[dry] = 0.0
    qy_new[dry] = 0.0

    return h_new, qx_new, qy_new

print('✓ Solver NSWE Lax-Friedrichs definido')

In [ ]:
# Celda 6/8 — Parámetros de la simulación
DT           = 1.0     # s  (CFL ≤ dx/(√2·c_max) ≈ 1.6 s → seguro)
NUM_SECS     = 1800.0  # s  (30 minutos)
OUTPUT_EVERY = 60.0    # s  (snapshot cada 60 s)

N_STEPS   = int(NUM_SECS / DT)
OUT_STEPS = int(OUTPUT_EVERY / DT)

c_max = float(np.sqrt(G_GRAV * np.maximum(-b, 1).max()))
cfl   = c_max * DT / min(dx, dy)
print(f'Grilla: {ny}×{nx} | dt={DT}s | {N_STEPS} pasos | {N_STEPS//OUT_STEPS} snapshots')
print(f'c_max={c_max:.0f} m/s | CFL={cfl:.3f} (debe ser < 0.71)')

In [ ]:
# Celda 7/8 — Ejecutar simulación (< 5 min en CPU)
h  = h_init.copy()
qx = qx_init.copy()
qy = qy_init.copy()

print('Iniciando simulación NSWE...\n')
t0_wall = time.time()

for step in range(N_STEPS + 1):
    t_sim = step * DT

    if step % OUT_STEPS == 0:
        # Guardar snapshot en orientación del DEM almacenado (row0=N, col0=E)
        h_save = h[::-1, ::-1].astype(np.float32)
        out_path = OUTPUT_DIR / f'h-{t_sim:.1f}.np'
        with open(str(out_path), 'wb') as fout:
            np.save(fout, h_save)

        elapsed = time.time() - t0_wall
        eta_ocean = float(np.max(np.abs(h[ocean_mask] + b[ocean_mask])))
        if step > 0:
            remain = elapsed / step * (N_STEPS - step)
            eta_str = f'~{remain:.0f}s restantes'
        else:
            eta_str = ''
        print(f't={t_sim/60:4.0f} min | η_max(mar)={eta_ocean:.2f}m'
              f' | {elapsed:.0f}s transcurridos {eta_str}')

    if step < N_STEPS:
        h, qx, qy = nswe_lf_step(h, qx, qy, b, dx, dy, DT)

total = time.time() - t0_wall
print(f'\n✓ Simulación completada en {total:.1f}s ({total/60:.1f} min)')
print(f'Archivos en: {OUTPUT_DIR}')

In [ ]:
# Celda 8/8 — Verificar salida y vista previa
import glob

output_files = sorted(glob.glob(str(OUTPUT_DIR / 'h-*.np')))
if not output_files:
    print('⚠ Sin archivos de salida en:', OUTPUT_DIR)
else:
    total_mb = sum(Path(f).stat().st_size for f in output_files) / 1e6
    print(f'✓ {len(output_files)} snapshots | {total_mb:.1f} MB')
    print(f'  {Path(output_files[0]).name} → {Path(output_files[-1]).name}')

    # Vista previa en t ≈ 15 min
    t_files = {float(Path(f).stem[2:]): f for f in output_files}
    t_mid   = min(t_files, key=lambda t: abs(t - 900.0))
    h_arr   = np.load(t_files[t_mid])            # shape (ny, nx), orientación DEM

    with rasterio.open(str(DEM_PATH)) as src:
        dem_vis = src.read(1).astype(np.float32)

    eta_vis = h_arr + np.minimum(dem_vis, 0)     # elevación superficie libre
    # Inundación sobre tierra: h por encima del mínimo húmedo
    inund   = np.where(dem_vis >= 0, np.maximum(h_arr - EPS_DEPTH, 0.0), 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    im1 = axes[0].imshow(eta_vis, cmap='RdBu_r', vmin=-3, vmax=3)
    plt.colorbar(im1, ax=axes[0], label='η (m sobre NMM)')
    axes[0].set_title(f't = {t_mid:.0f}s ({t_mid/60:.0f} min) — Superficie libre')
    im2 = axes[1].imshow(inund,   cmap='YlOrRd',  vmin=0,  vmax=5)
    plt.colorbar(im2, ax=axes[1], label='Inundación (m)')
    axes[1].set_title('Profundidad de agua sobre terreno')
    plt.suptitle('Tsunami Tumaco 1979 — Vista previa t=15 min')
    plt.tight_layout()
    plt.savefig(str(WORK_DIR / 'preview_t900s.png'), dpi=150)
    plt.show()
    print(f'Inundación máxima: {inund.max():.2f} m')